*[cell 0]*

# LVL_0 Report

Descriptive tables on one stamped LVL_0 handoff, pinned as `LVL0_HANDOFF` in `[cell 1]`. The report reads the handoff,
its run log (`lv0_out_<ts>_parameters.json`: the intake gates and the drift models LVL_0 fitted) and the group-level
decay fits the LVL0 notebook exports beside the handoff (`lv0_out_<ts>_decay_fits.csv`). It fits nothing.

**Conditions.** The manuscript calls the two conditions cohorts: T2 = Cohort A (CA), T3 = Cohort B (CB). The code,
file names, exports and the final report keep T2 and T3; only `make_replication_figures.ipynb` prints CA and CB.

| Table | Content | Export |
|---|---|---|
| 0.2 | Session duration (summed response times) by condition and session | `session_duration` |
| 0.2b | Session duration: T2 vs T3 and S1 vs S2 tests | `session_duration_tests` |
| 0.4 | Responses outside the RT limits | `rt_limits` |
| 0.5 | Drift models: the model per span and its fit per scale | `drift_models` |
| 0.6 | Drift-model selection: mean R² of every candidate per span | `model_selection` |

Each table is written to `LVL_OUTS` as `<handoff stem>_report_<export>.csv`; the final report prints them. The
decay-curve figures are drawn by `make_replication_figures.ipynb` from the LVL0 notebook's exports; the sample
composition is `demographic_report.ipynb`'s.

In [1]:
# [cell 1]
# ===== INPUTS: set these before running =====
LVL0_HANDOFF = "lv0_out_20261106_180438.csv"   # the LVL0 output in LVL_OUTS, e.g. "lv0_out_20261005_101500.csv"

# =============================================================================
# IMPORTS, INPUT, PATHS
# =============================================================================
import json, re, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 200)

# ---- locations: everything is inside /home/sjsuss626/EMST_files/sjs_dissertation ----------------
LVL_OUTS = Path("/home/sjsuss626/EMST_files/sjs_dissertation/dervatives/dynamic_qc/lvl_outs")   # the handoff, its exports; this report's tables

if LVL0_HANDOFF is None:
    raise SystemExit("LVL0_HANDOFF is not set (top of this cell). Name one LVL_0 handoff; "
                     f"available in {LVL_OUTS}: {sorted(p.name for p in LVL_OUTS.glob('lv0_out_*.csv'))}")
LV0_PATH = LVL_OUTS / LVL0_HANDOFF
_m0 = re.match(r"lv0_out_(\d{8}_\d{6})", LV0_PATH.name)
if not _m0 or LV0_PATH.suffix != ".csv" or not LV0_PATH.is_file():
    raise SystemExit(f"LVL0_HANDOFF = {LVL0_HANDOFF!r} is not an existing LVL_0 handoff "
                     "(lv0_out_<YYYYmmdd_HHMMSS>[...].csv)")
# The handoff's LVL_0 run log (configuration, drift models and their fits): the one file of the same stamp beside it.
_recs = sorted(LV0_PATH.parent.glob(f"lv0_out_{_m0.group(1)}*_parameters.json"))
if len(_recs) != 1:
    raise SystemExit(f"expected exactly one LVL_0 run log lv0_out_{_m0.group(1)}*_parameters.json beside "
                     f"{LV0_PATH.name}, found {[p.name for p in _recs]}")
LVL0_RUN_LOG = json.load(open(_recs[0]))
LVL0_PARAMS = LVL0_RUN_LOG["parameters"]
# The group-level decay fits, exported by the LVL0 notebook beside the handoff.
DECAY_FITS_PATH = LV0_PATH.with_name(f"{LV0_PATH.stem}_decay_fits.csv")
if not DECAY_FITS_PATH.is_file():
    raise SystemExit(f"{DECAY_FITS_PATH.name} is missing: the LVL0 notebook's export cell writes it beside the handoff")
OUT_STEM = LVL_OUTS / f"{LV0_PATH.stem}_report"

PR_STIM_DROP  = LVL0_PARAMS["pr_stim_drop"]     # first N stimuli of each session (practice)
MIN_RT_THRESH = LVL0_PARAMS["min_rt_thresh"]    # seconds
MAX_RT_THRESH = LVL0_PARAMS["max_rt_thresh"]    # seconds
REPORT_TABLES = {}


def export(key, frame, title):
    """Write one table to LVL_OUTS as <handoff stem>_report_<key>.csv and show it."""
    REPORT_TABLES[key] = frame
    frame.to_csv(f"{OUT_STEM}_{key}.csv", index=False)
    print(title)
    display(frame)


def bh(p):
    """Benjamini-Hochberg q-values; NaN stays NaN."""
    p = np.asarray(p, float); q = np.full(len(p), np.nan); ok = np.isfinite(p)
    if ok.any():
        q[ok] = multipletests(p[ok], method="fdr_bh")[1]
    return q


print(f"LVL_0 handoff : {LV0_PATH}")
print(f"LVL_0 run log : {_recs[0].name}  (pr_stim_drop={PR_STIM_DROP}, min_rt_thresh={MIN_RT_THRESH}, "
      f"max_rt_thresh={MAX_RT_THRESH})")
print(f"decay fits    : {DECAY_FITS_PATH.name}")
print(f"outputs       : {OUT_STEM}_*.csv")

# Interpreter and package versions of this run.
import sys as _vsys, platform as _vplatform
from importlib import metadata as _vmeta


def _pkg_version(p):
    try:
        return _vmeta.version(p)
    except _vmeta.PackageNotFoundError:
        return "not installed"


print(f"interpreter: {_vsys.executable} (Python {_vplatform.python_version()})")
print("packages: " + ", ".join(f"{p} {_pkg_version(p)}" for p in (
    "numpy", "pandas", "scipy", "scikit-learn", "statsmodels", "hmmlearn", "ruptures", "joblib", "matplotlib")))


LVL_0 handoff : /home/sjsuss626/EMST_files/sjs_dissertation/dervatives/dynamic_qc/lvl_outs/lv0_out_20261106_180438.csv
LVL_0 run log : lv0_out_20261106_180438_parameters.json  (pr_stim_drop=3, min_rt_thresh=0.15, max_rt_thresh=90)
decay fits    : lv0_out_20261106_180438_decay_fits.csv
outputs       : /home/sjsuss626/EMST_files/sjs_dissertation/dervatives/dynamic_qc/lvl_outs/lv0_out_20261106_180438_report_*.csv
interpreter: /usr/bin/python3 (Python 3.10.12)
packages: numpy 1.26.4, pandas 2.2.1, scipy 1.12.0, scikit-learn 1.7.2, statsmodels 0.14.6, hmmlearn 0.3.3, ruptures 1.1.10, joblib 1.5.3, matplotlib 3.10.8


In [2]:
# [cell 2]
# =============================================================================
# LOAD
# =============================================================================
# Every response of the analysed sample (the LVL_0 handoff keeps all of them; the practice stimuli and the RT limits
# are applied only where a table says so).
lv0 = pd.read_csv(LV0_PATH, usecols=["id", "condition", "session", "stim_order", "rt_sec"], low_memory=False)
print(f"LVL_0 handoff: {lv0.id.nunique()} participants, {len(lv0.drop_duplicates(['id', 'session']))} sessions, "
      f"{len(lv0):,} responses")
DECAY_FITS = pd.read_csv(DECAY_FITS_PATH)
DRIFT_FITS = pd.DataFrame(LVL0_RUN_LOG["statistical_results"]["drift_fit_parameters"])
SPAN = {"Overall (Concatenated)": "Overall", "Session 1": "S1", "Session 2": "S2"}
# the decay-fit export's selected model per span is the model LVL_0 fitted (its run log)
_sel = DECAY_FITS[DECAY_FITS["selected"]].set_index("span")["model"].to_dict()
_run = DRIFT_FITS.assign(span=DRIFT_FITS.session_label.map(SPAN)).groupby("span").model_name.agg(set).to_dict()
if {s: {m} for s, m in _sel.items()} != _run:
    raise SystemExit(f"the decay-fit export's models {_sel} differ from the LVL_0 run log's {_run}: "
                     "re-run the LVL0 notebook's export cell on this handoff")

LVL_0 handoff: 282 participants, 553 sessions, 552,281 responses


In [3]:
# [cell 3]
# =============================================================================
# TABLES 0.2 / 0.2b: SESSION DURATION
# =============================================================================
# Duration = the sum of the raw response times of every response in the session (five scales per stimulus, before the
# RT limits), in minutes; the time between responses is not recorded. T2 vs T3 on participant means (Mann-Whitney U);
# S1 vs S2 among participants with both sessions (Wilcoxon signed-rank); q = BH over the four tests.
dur = lv0.groupby(["id", "condition", "session"]).rt_sec.sum().div(60).rename("minutes").reset_index()
_w = dur.pivot_table(index=["id", "condition"], columns="session", values="minutes").reset_index()
_pm = dur.groupby(["id", "condition"]).minutes.mean().reset_index()
_mw = stats.mannwhitneyu(_pm.minutes[_pm.condition == "T2"], _pm.minutes[_pm.condition == "T3"])
_wx = {}
for lab, g in (("T2", _w[_w.condition == "T2"]), ("T3", _w[_w.condition == "T3"]), ("pooled", _w)):
    g = g.dropna(subset=[1, 2]); _wx[lab] = (stats.wilcoxon(g[1], g[2]), len(g))
_qd = bh([_mw.pvalue] + [v[0].pvalue for v in _wx.values()])
_med = lambda x: f"{x.median():.1f} [{x.quantile(.25):.1f}, {x.quantile(.75):.1f}]"
_msd = lambda x: f"{x.mean():.1f} ({x.std():.1f})"
rows = [{"Group": f"{cd}, S{s}", "Sessions": len(g), "Duration, min: Mdn [Q1, Q3]": _med(g.minutes),
         "Duration, min: M (SD)": _msd(g.minutes)}
        for cd in ("T2", "T3") for s in (1, 2) for g in [dur[(dur.condition == cd) & (dur.session == s)]]]
rows += [{"Group": f"{cd} (participant mean of sessions)", "Sessions": int((dur.condition == cd).sum()),
          "Duration, min: Mdn [Q1, Q3]": _med(_pm.minutes[_pm.condition == cd]),
          "Duration, min: M (SD)": _msd(_pm.minutes[_pm.condition == cd])} for cd in ("T2", "T3")]
rows += [{"Group": "All sessions", "Sessions": len(dur), "Duration, min: Mdn [Q1, Q3]": _med(dur.minutes),
          "Duration, min: M (SD)": _msd(dur.minutes)}]
export("session_duration", pd.DataFrame(rows),
       "Table 0.2. Session duration (summed response times) by condition and session")
export("session_duration_tests", pd.DataFrame(
    [{"Comparison": "T2 vs T3 (participant means; Mann–Whitney U)", "n": len(_pm), "Statistic": _mw.statistic,
      "p": _mw.pvalue, "q": _qd[0]}] +
    [{"Comparison": f"S1 vs S2, {lab} (Wilcoxon signed-rank)", "n": v[1], "Statistic": v[0].statistic,
      "p": v[0].pvalue, "q": _qd[i + 1]} for i, (lab, v) in enumerate(_wx.items())]),
    "Table 0.2b. Session duration: T2 vs T3 and S1 vs S2 tests")

Table 0.2. Session duration (summed response times) by condition and session


,Group,Sessions,"Duration, min: Mdn [Q1, Q3]","Duration, min: M (SD)"
0,"T2, S1",136,"52.0 [43.9, 66.9]",56.8 (23.1)
1,"T2, S2",136,"38.4 [28.5, 46.9]",38.3 (13.6)
2,"T3, S1",141,"51.4 [42.3, 68.0]",56.1 (22.1)
3,"T3, S2",140,"37.3 [27.7, 50.4]",40.3 (19.2)
4,T2 (participant mean of sessions),272,"46.1 [36.9, 56.3]",47.4 (16.8)
5,T3 (participant mean of sessions),281,"46.1 [34.3, 58.8]",48.3 (19.8)
6,All sessions,553,"45.5 [33.8, 58.1]",47.9 (21.6)


Table 0.2b. Session duration: T2 vs T3 and S1 vs S2 tests


,Comparison,n,Statistic,p,q
0,T2 vs T3 (participant means; Mann–Whitney U),282,10044.0,8.781146e-01,8.781146e-01
1,"S1 vs S2, T2 (Wilcoxon signed-rank)",133,54.0,4.840432e-23,6.453910e-23
2,"S1 vs S2, T3 (Wilcoxon signed-rank)",138,132.0,3.717639e-23,6.453910e-23
3,"S1 vs S2, pooled (Wilcoxon signed-rank)",271,364.0,1.849543e-44,7.398174e-44


In [4]:
# [cell 4]
# =============================================================================
# TABLE 0.4: RESPONSES OUTSIDE THE RT LIMITS
# =============================================================================
# The RT limits set a response's RT to missing at LVL_1 (every response); the practice stimuli are left out of the
# LVL_0 drift fits only.
_pr, _hi, _lo = int(PR_STIM_DROP), float(MAX_RT_THRESH), float(MIN_RT_THRESH)
rows = []
for lab, d_ in (("all responses", lv0), (f"excluding the first {_pr} (practice) stimuli of each session",
                                         lv0[lv0.stim_order > _pr])):
    rows.append({"Responses": lab, "n": len(d_),
                 f"RT > {_hi:g} s, n (%)": f"{int((d_.rt_sec > _hi).sum()):,} ({100 * (d_.rt_sec > _hi).mean():.2f})",
                 f"RT < {_lo:g} s, n (%)": f"{int((d_.rt_sec < _lo).sum()):,} ({100 * (d_.rt_sec < _lo).mean():.2f})"})
export("rt_limits", pd.DataFrame(rows), "Table 0.4. Preprocessing: RTs outside the limits")

Table 0.4. Preprocessing: RTs outside the limits


,Responses,n,"RT > 90 s, n (%)","RT < 0.15 s, n (%)"
0,all responses,552281,220 (0.04),"11,080 (2.01)"
1,excluding the first 3 (practice) stimuli of ea...,543996,216 (0.04),"11,004 (2.02)"


In [5]:
# [cell 5]
# =============================================================================
# TABLES 0.5 / 0.6: DRIFT MODELS
# =============================================================================
# Table 0.5: per span, the model LVL_0 fitted and its R^2 per scale (LVL_0's fit to the median RT in seconds, from the
# run log), with the R^2 of the same model fitted to the median log RT (the LVL0 notebook's decay-fit export: the
# curves of the decay-curve figure). Table 0.6: the mean R^2 over the scales of every candidate, per span; the highest
# is the model used.
rows = []
for r in DRIFT_FITS.itertuples():
    sp = SPAN[r.session_label]
    lg = DECAY_FITS[(DECAY_FITS["table"] == "figure fit") & (DECAY_FITS["transform"] == "log")
                    & (DECAY_FITS["span"] == sp) & (DECAY_FITS["scale"] == r.scale)]
    rows.append({"Span": r.session_label, "Scale": r.scale, "Model": r.model_name,
                 "R² (LVL_0 fit, median RT in s)": r.r_squared,
                 "R² (log-scale fit)": lg["r_squared"].iloc[0] if len(lg) else np.nan})
export("drift_models", pd.DataFrame(rows), "Table 0.5. Drift models: the selected model per span and its fit per scale")
_ms = DECAY_FITS[DECAY_FITS["table"] == "model selection"]
export("model_selection", _ms.pivot_table(index="model", columns="span", values="r_squared").reset_index()
       .rename(columns={"model": "Model"}), "Table 0.6. Drift-model selection: mean R² of every candidate, per span")

Table 0.5. Drift models: the selected model per span and its fit per scale


,Span,Scale,Model,"R² (LVL_0 fit, median RT in s)",R² (log-scale fit)
0,Overall (Concatenated),arousal,log_w_linfag,0.975032,0.964228
1,Overall (Concatenated),emotion,log_w_linfag,0.963446,0.952319
2,Overall (Concatenated),similarity,log_w_linfag,0.964595,0.955684
3,Overall (Concatenated),threat,log_w_linfag,0.952562,0.948574
4,Overall (Concatenated),valence,log_w_linfag,0.956751,0.951121
5,Session 1,arousal,log_w_linfag,0.979142,0.976066
6,Session 1,emotion,log_w_linfag,0.960971,0.954448
7,Session 1,similarity,log_w_linfag,0.964627,0.959114
8,Session 1,threat,log_w_linfag,0.928288,0.923018
9,Session 1,valence,log_w_linfag,0.943043,0.938678


Table 0.6. Drift-model selection: mean R² of every candidate, per span


span,Model,Overall,S1,S2
0,exp,0.936110,0.931605,0.771194
1,exp_w_linfag,0.959578,0.951136,0.780032
2,linear,0.786592,0.818124,0.759537
3,log,0.931230,0.920915,0.684360
4,log_w_linfag,0.962477,0.955214,0.779080
